# X-OPT - REOPTIMIZATION WITH STRUCTURAL AND FREQUENCY CANDIDATES

The experiment solves the first 20 OR-Library p-median instances, extracts the interpretable structures given by the `highest k-core` and the `densest subgraph` from the co-occurrence graph built from the long-term memory, perturbs the edge costs, and reoptimizes the perturbed instance using restricted candidate sets.

The evaluated variants are:

- `baseline`: unrestricted p-median on the perturbed distance matrix;
- `highest_k_core`: candidate facilities restricted to the maximum k-core;
- `densest_subgraph`: candidate facilities restricted to the densest subgraph;
- `nonzero_frequency`: all facilities with absolute frequency greater than zero in the analyzed memory;
- `top_frequency_k_core_size`: the most frequent facilities, using the same number of items as the k-core;
- `top_frequency_densest_size`: the most frequent facilities, using the same number of items as the densest subgraph.

Frequencies are computed from the same analyzed memory used to construct the structures: the top `10%` of the LTM sorted by cost, unless changed by the parameters below.

> Obs.: This notebook extends the workflow from `4.0 Optimizing Perturbed Instances.ipynb`.

### SETUP

In [1]:
from __future__ import annotations

import gc
import sys

import numpy  as np
import pandas as pd

from tqdm.auto       import tqdm
from pathlib         import Path
from IPython.display import display
from time            import perf_counter
from mip             import BINARY, Model, OptimizationStatus, minimize, xsum


pd.set_option("display.max_columns" , None)
pd.set_option("display.max_colwidth", 140 )

In [2]:
from lib.paths     import find_project_root      , \
                          instance_sort_key      , \
                          canonical_instance_name

from lib.instances import list_orlibrary_instances, \
                          read_instance_metadata  , \
                          load_best_known_costs_to_dict_id

from lib.graph     import read_orlibrary_graph     , \
                          all_pairs_shortest_paths , \
                          build_top_ltm            , \
                          build_cooccurrence_matrix, \
                          build_unweighted_graph   , \
                          total_edge_count         , \
                          total_edge_weight

from lib.explain   import densest_subgraph_greedy     , \
                          extract_highest_k_core_nodes

from lib.mip       import extract_open_facilities_candidates

from lib.metrics   import compute_solution_cost   , \
                          gap_to_reference_percent, \
                          improvement_percent

from lib.utils     import finite_or_none

In [3]:
PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


print(f"Project root is {PROJECT_ROOT}!")

Project root is /home/rei-luisinho/xopt!


In [4]:
import pymedian

### EXPERIMENT CONFIGURATION

In [5]:
INSTANCES_DIR = PROJECT_ROOT  / "instances"
PMEDOPT_PATH  = INSTANCES_DIR / "pmedopt.txt"
OUTPUT_DIR    = PROJECT_ROOT  / "notebooks" / "experiments_sbpo" / "artifacts"

RAW_RESULTS_CSV       = OUTPUT_DIR / "reoptimization_frequency_candidates_raw.csv"
SUMMARY_RESULTS_CSV   = OUTPUT_DIR / "reoptimization_frequency_candidates_summary.csv"
AGGREGATE_RESULTS_CSV = OUTPUT_DIR / "reoptimization_frequency_candidates_aggregate.csv"
FAILURES_CSV          = OUTPUT_DIR / "reoptimization_frequency_candidates_failures.csv"

ALL_INSTANCE_NAMES = list_orlibrary_instances(INSTANCES_DIR)
INSTANCE_NAMES     = ALL_INSTANCE_NAMES[:20]

RESTARTS       = 8
MAX_ITER       = 25
FACTOR         = 0.25
DETAILS_FORMAT = "binary"

TOP_FRACTION   = 0.1

TIME_LIMIT_SECONDS = 400
PERTURBATION_DELTA = 0.10
GLOBAL_SEED        = 42

SAVE_RESULTS_CSV = True

BEST_KNOWN_COSTS = load_best_known_costs_to_dict_id(PMEDOPT_PATH)


if len(INSTANCE_NAMES) != 20:
    raise ValueError(
        f"Expected 20 selected OR-Library instances, found {len(INSTANCE_NAMES)}."
    )

In [6]:
print(f"Instances discovered : {len(ALL_INSTANCE_NAMES)}")
print(f"Instances selected   : {len(INSTANCE_NAMES    )}")
print(f"Selected range       : {INSTANCE_NAMES[0]} ... {INSTANCE_NAMES[-1]}")

Instances discovered : 40
Instances selected   : 20
Selected range       : pmed1.txt ... pmed20.txt


In [7]:
print(f"Metaheuristic parameters   : restarts={RESTARTS}, max_iter={MAX_ITER}, factor={FACTOR}")
print(f"Reoptimization time limit  : {TIME_LIMIT_SECONDS}s"   )
print(f"Top fraction kept from LTM : {      TOP_FRACTION:.0%}")
print(f"Perturbation delta         : {PERTURBATION_DELTA:.1%}")

Metaheuristic parameters   : restarts=8, max_iter=25, factor=0.25
Reoptimization time limit  : 400s
Top fraction kept from LTM : 10%
Perturbation delta         : 10.0%


In [8]:
selected_instances_df = pd.DataFrame(
    [
        {
            "instance"    :      canonical_instance_name(instance_name)      ,
            "instance_id" : Path(canonical_instance_name(instance_name)).stem,

            "best_known_cost" : BEST_KNOWN_COSTS.get(
                Path(canonical_instance_name(instance_name)).stem, np.nan
            ),

            **read_instance_metadata(
                INSTANCES_DIR / canonical_instance_name(instance_name)
            ),
        }
        for instance_name in INSTANCE_NAMES
    ]
)

selected_instances_df["instance_order"] = selected_instances_df["instance"].map(
    lambda value: instance_sort_key(value)[0]
)

selected_instances_df = (
    selected_instances_df.sort_values(["instance_order", "instance"])
                         .drop       (columns="instance_order")
                         .reset_index(drop   =True            )
)


display(selected_instances_df)

,instance,instance_id,best_known_cost,n,p
0,pmed1.txt,pmed1,5819.0,100,5
1,pmed2.txt,pmed2,4093.0,100,10
2,pmed3.txt,pmed3,4250.0,100,10
3,pmed4.txt,pmed4,3034.0,100,20
4,pmed5.txt,pmed5,1355.0,100,33
5,pmed6.txt,pmed6,7824.0,200,5
6,pmed7.txt,pmed7,5631.0,200,10
7,pmed8.txt,pmed8,4445.0,200,20
8,pmed9.txt,pmed9,2734.0,200,40
9,pmed10.txt,pmed10,1255.0,200,67


### PERTURBATION

In [9]:
def perturb_adjacency(
    adjacency: list[list[tuple[int, int]]],
    *,
    delta             : float,
    seed              : int  ,
    affected_fraction : float = 0.10,
) -> tuple[list[list[tuple[int, int]]], dict[str, float | int]]:
    rng = np.random.default_rng(seed)
    n   = len(adjacency)

    perturbed = [[] for _ in range(n)]

    unique_edges = []
    for u, neighbors in enumerate(adjacency):
        for v, weight in neighbors:
            if u < v:
                unique_edges.append((u, v, weight))

    edge_count     = len(unique_edges)
    affected_count = max(1, int(round(edge_count * affected_fraction)))

    affected_idx = set(
        rng.choice(edge_count, size=affected_count, replace=False)
    )

    multipliers = []

    for idx, (u, v, weight) in enumerate(unique_edges):
        if idx in affected_idx:
            multiplier = 1.0 + float(rng.uniform(-delta, delta))
            new_weight = max(1, int(round(weight * multiplier)))

            multipliers.append(multiplier)
        else:
            new_weight = weight

        perturbed[u].append((v, new_weight))
        perturbed[v].append((u, new_weight))

    for neighbors in perturbed:
        neighbors.sort()

    multipliers_array = np.asarray(multipliers, dtype=float)

    return perturbed, {
        "perturbation_edge_count"          : edge_count    ,
        "perturbation_affected_edge_count" : affected_count,
        "perturbation_affected_fraction"   : affected_count / edge_count     if edge_count             else 0.0,
        "perturbation_mean_multiplier"     : float(multipliers_array.mean()) if len(multipliers_array) else 1.0,
    }

### STRUCTURES AND FREQUENCIES

In [10]:
def best_facilities_from_summary(
    summary: dict[str, object]
) -> tuple[int, ...]:
    return tuple(
        sorted(
            int(value) - 1
            for value in summary["tspmed_facilities"]
        )
    )


def top_q_facilities_by_frequency(
    absolute_frequency: np.ndarray,
    q                 : int       ,
) -> set[int]:
    q = max(0, min(int(q), int(absolute_frequency.size)))

    if q == 0:
        return set()

    order = np.lexsort(
        (
             np.arange (absolute_frequency.size      ),
            -np.asarray(absolute_frequency, dtype=int),
        )
    )

    return set(int(value) for value in order[:q].tolist())


def nonzero_facilities_by_frequency(absolute_frequency: np.ndarray) -> set[int]:
    frequencies = np.asarray(absolute_frequency, dtype=int)

    return set(
        int(value)
        for value in np.flatnonzero(frequencies > 0).tolist()
    )


def set_recall(candidate_nodes: set[int], reference_nodes: set[int]) -> float:
    if not reference_nodes:
        return 0.0

    return len(candidate_nodes.intersection(reference_nodes)) / len(reference_nodes)


def extract_structures_and_frequency_candidates(
    summary      : dict[str, object],
    details      : dict[str, object],
    *,
    top_fraction : float,
) -> dict[str, object]:
    long_term_memory = details["long_term_memory"]

    if not long_term_memory:
        raise ValueError("Long term memory is empty.")

    analysis_ltm, matrix, costs = build_top_ltm            (long_term_memory, top_fraction)
    adjacency                   = build_cooccurrence_matrix(matrix   )
    unweighted_graph            = build_unweighted_graph   (adjacency)

    absolute_frequency          = np.asarray(matrix, dtype=int).sum(axis=0)
    nonzero_frequency_nodes     = nonzero_facilities_by_frequency(absolute_frequency)

    _, k_core_max_level, highest_k_core_nodes = extract_highest_k_core_nodes(unweighted_graph)
    densest_nodes      , densest_density      = densest_subgraph_greedy     (adjacency, min_size=max(3, int(summary["p"])))

    top_frequency_k_core_size  = top_q_facilities_by_frequency(absolute_frequency, len(highest_k_core_nodes))
    top_frequency_densest_size = top_q_facilities_by_frequency(absolute_frequency, len(densest_nodes       ))

    best_facilities = best_facilities_from_summary(summary)
    best_set        = set(best_facilities)

    return {
        "best_facilities" : best_facilities              ,
        "best_cost"       : float(summary["tspmed_cost"]),

        "memory_size"        : len  (long_term_memory),
        "top_solution_count" : len  (analysis_ltm    ),
        "top_cost_cutoff"    : float(costs.max()     ),
        "cooccurrence_edges" : total_edge_count(adjacency),

        "absolute_frequency"         : absolute_frequency     ,
        "nonzero_frequency_nodes"    : nonzero_frequency_nodes,
        "nonzero_frequency_count"    : len(nonzero_frequency_nodes),
        "nonzero_frequency_fraction" : len(nonzero_frequency_nodes) / max(1, int(summary["n"])),

        "highest_k_core_nodes"      : highest_k_core_nodes,
        "k_core_max_level"          : int(k_core_max_level    ),
        "k_core_candidate_count"    : len(highest_k_core_nodes),
        "k_core_candidate_fraction" : len(highest_k_core_nodes) / max(1, int(summary["n"])),
        "k_core_best_set_recall"    : set_recall(highest_k_core_nodes, best_set),

        "densest_nodes"                       : densest_nodes         ,
        "densest_subgraph_density"            : float(densest_density),
        "densest_subgraph_candidate_count"    : len(densest_nodes),
        "densest_subgraph_candidate_fraction" : len(densest_nodes) / max(1, int(summary["n"])),
        "densest_subgraph_best_set_recall"    : set_recall(densest_nodes, best_set),

        "top_frequency_k_core_size_nodes"           : top_frequency_k_core_size     ,
        "top_frequency_k_core_size_count"           : len(top_frequency_k_core_size),
        "top_frequency_k_core_size_best_set_recall" : set_recall(top_frequency_k_core_size, best_set),
        "top_frequency_k_core_k_core_intersection"  : len(
            top_frequency_k_core_size.intersection(highest_k_core_nodes)
        ),

        "top_frequency_densest_size_nodes"           : top_frequency_densest_size     ,
        "top_frequency_densest_size_count"           : len(top_frequency_densest_size),
        "top_frequency_densest_size_best_set_recall" : set_recall(top_frequency_densest_size, best_set),
        "top_frequency_densest_densest_intersection" : len(
            top_frequency_densest_size.intersection(densest_nodes)
        ),
    }

### REOPTIMIZATION

In [11]:
def build_pmedian_model(
    distances         : np.ndarray,
    p                 : int       ,
    *,
    allowed_facilities: list[int] | tuple[int, ...] | set[int] | None = None,
) -> tuple[Model, list[list], list, list[int]]:
    if distances.ndim != 2 or distances.shape[0] != distances.shape[1]:
        raise ValueError("Distances must be a square 2D array.")

    n = distances.shape[0]
    if not (1 <= p <= n):
        raise ValueError("P must satisfy 1 <= p <= n.")

    if allowed_facilities is None:
        candidate_facilities = list(range(n))
    else:
        candidate_facilities = sorted(
            {int(facility) for facility in allowed_facilities}
        )

    if len(candidate_facilities) < p:
        raise ValueError(
            f"Candidate facility pool is too small: {len(candidate_facilities)} < {p}."
        )

    model = Model(solver_name="CBC")
    model.verbose = 0

    y = [
        model.add_var(var_type=BINARY)
        for _ in candidate_facilities
    ]

    x: list[list] = []

    for i in range(n):
        x_row = [
            model.add_var(var_type=BINARY)
            for _ in candidate_facilities
        ]
        x.append(x_row)

        model.add_constr(xsum(x_row) == 1)

        for pos in range(len(candidate_facilities)):
            model.add_constr(x_row[pos] <= y[pos])

    model.add_constr(xsum(y) == p)

    model.objective = minimize(
        xsum(
            float(distances[i, facility]) * x[i][pos]
            for i in range(n)
            for pos, facility in enumerate(candidate_facilities)
        )
    )

    return model, x, y, candidate_facilities


def optimize_model(
    model              : Model     ,
    time_limit_seconds : int | None,
) -> OptimizationStatus:
    if time_limit_seconds is None:
        return model.optimize()

    return model.optimize(max_seconds=float(time_limit_seconds))

In [12]:
def solve_reoptimization_variant(
    *,
    variant            : str       ,
    variant_order      : int       ,
    distances          : np.ndarray,
    p                  : int       ,
    time_limit_seconds : int | None,
    allowed_facilities : list[int] | tuple[int, ...] | set[int] | None = None,
) -> dict[str, object]:
    n = distances.shape[0]

    candidate_facilities = (
        list(range(n))
        if   allowed_facilities is None
        else sorted({int(value) for value in allowed_facilities})
    )

    row = {
        "variant"                     : variant      ,
        "variant_order"               : variant_order,

        "candidate_facility_count"    : len(candidate_facilities),
        "candidate_facility_fraction" : len(candidate_facilities) / max(1, n),

        "objective_value"             : np.nan,
        "model_build_seconds"         : np.nan,
        "solve_seconds"               : np.nan,
        "total_variant_seconds"       : np.nan,
        "open_facilities_count"       : 0 ,
        "open_facilities"             : "",
        "status"                      : "ERROR",
        "error_message"               : None   ,
    }

    if len(candidate_facilities) < p:
        row.update(
            {
                "status"        : "INFEASIBLE_POOL",
                "error_message" : f"Candidate facility pool smaller than p: {len(candidate_facilities)} < {p}.",
            }
        )

        return row

    build_started_at = perf_counter()

    try:
        model, _, y, candidate_facilities = build_pmedian_model(
            distances,
            p        ,
            allowed_facilities=candidate_facilities,
        )

        build_seconds = perf_counter() - build_started_at

        solve_started_at = perf_counter()
        status           = optimize_model(model, time_limit_seconds)
        solve_seconds    = perf_counter() - solve_started_at

        has_incumbent = status in {
            OptimizationStatus.OPTIMAL ,
            OptimizationStatus.FEASIBLE,
        }

        solver_objective = finite_or_none(
            model.objective_value if has_incumbent else None
        )

        open_facilities: list[int] = []
        validated_objective        = None

        if has_incumbent:
            open_facilities = extract_open_facilities_candidates(
                candidate_facilities, y
            )

            if len(open_facilities) != p:
                raise ValueError(
                    f"Expected {p} open facilities, but recovered {len(open_facilities)}."
                )

            validated_objective = compute_solution_cost(
                distances, open_facilities
            )

            if solver_objective is not None and validated_objective is not None:
                if abs(solver_objective - validated_objective) > 1e-4:
                    raise ValueError(
                        "Solver objective and validated objective do not match: "
                        f"{solver_objective} vs {validated_objective}."
                    )

        objective_value = (
            validated_objective
            if   validated_objective is not None
            else solver_objective
        )

        row.update(
            {
                "objective_value"       : objective_value if objective_value is not None else np.nan,

                "model_build_seconds"   : build_seconds,
                "solve_seconds"         : solve_seconds,
                "total_variant_seconds" : build_seconds + solve_seconds,

                "open_facilities_count" : len(open_facilities),
                "open_facilities"       : " ".join(
                    str(facility + 1) for facility in open_facilities
                ),

                "status"        : getattr(status, "name", str(status)),
                "error_message" : None,
            }
        )
    except Exception as exc:
        row.update(
            {
                "status"              : "ERROR",
                "error_message"       : f"{type(exc).__name__}: {exc}"   ,
                "model_build_seconds" : perf_counter() - build_started_at,
            }
        )
    finally:
        for name in ["model", "y"]:
            if name in locals():
                del locals()[name]

        gc.collect()

    return row

### BENCHMARK EXECUTION

In [13]:
VARIANT_SPECS = [
    {"variant" : "baseline"                  , "variant_order" : 1, "allowed_key" : None                              },
    {"variant" : "highest_k_core"            , "variant_order" : 2, "allowed_key" : "highest_k_core_nodes"            },
    {"variant" : "densest_subgraph"          , "variant_order" : 3, "allowed_key" : "densest_nodes"                   },
    {"variant" : "nonzero_frequency"         , "variant_order" : 4, "allowed_key" : "nonzero_frequency_nodes"         },
    {"variant" : "top_frequency_k_core_size" , "variant_order" : 5, "allowed_key" : "top_frequency_k_core_size_nodes" },
    {"variant" : "top_frequency_densest_size", "variant_order" : 6, "allowed_key" : "top_frequency_densest_size_nodes"},
]

In [14]:
def build_pipeline_error_rows(
    instance_name   : str,
    instance_id     : str,
    metadata        : dict[str, int],
    best_known_cost : float | None  ,
    error_message   : str,
) -> list[dict[str, object]]:
    rows = []

    for spec in VARIANT_SPECS:
        rows.append(
            {
                "instance"        : instance_name,
                "instance_id"     : instance_id  ,
                "n"               : metadata.get("n", np.nan),
                "p"               : metadata.get("p", np.nan),
                "best_known_cost" : best_known_cost if best_known_cost is not None else np.nan,

                "metaheuristic_best_cost"       : np.nan,
                "metaheuristic_gap_percent"     : np.nan,
                "metaheuristic_runtime_seconds" : np.nan,

                "structure_extraction_runtime_seconds"  : np.nan,
                "perturbation_runtime_seconds"          : np.nan,
                "pre_reoptimization_seconds"            : np.nan,
                "previous_solution_perturbed_objective" : np.nan,
                "previous_solution_degradation_percent" : np.nan,

                "memory_size"        : np.nan,
                "top_solution_count" : np.nan,
                "top_cost_cutoff"    : np.nan,
                "cooccurrence_edges" : np.nan,

                "k_core_max_level"          : np.nan,
                "k_core_candidate_count"    : np.nan,
                "k_core_candidate_fraction" : np.nan,
                "k_core_best_set_recall"    : np.nan,

                "densest_subgraph_candidate_count"    : np.nan,
                "densest_subgraph_candidate_fraction" : np.nan,
                "densest_subgraph_density"            : np.nan,
                "densest_subgraph_best_set_recall"    : np.nan,

                "nonzero_frequency_count"                    : np.nan,
                "nonzero_frequency_fraction"                 : np.nan,
                "top_frequency_k_core_size_count"            : np.nan,
                "top_frequency_k_core_size_best_set_recall"  : np.nan,
                "top_frequency_k_core_k_core_intersection"   : np.nan,
                "top_frequency_densest_size_count"           : np.nan,
                "top_frequency_densest_size_best_set_recall" : np.nan,
                "top_frequency_densest_densest_intersection" : np.nan,

                "perturbation_delta"          : PERTURBATION_DELTA,
                "perturbation_seed"           : np.nan,
                "perturbation_edge_count"     : np.nan,
                "perturbation_mean_multiplier": np.nan,
                "variant"                     : spec["variant"      ],
                "variant_order"               : spec["variant_order"],

                "candidate_facility_count"          : np.nan,
                "candidate_facility_fraction"       : np.nan,
                "objective_value"                   : np.nan,
                "improvement_over_previous_percent" : np.nan,
                "gap_to_unrestricted_percent"       : np.nan,

                "model_build_seconds"   : np.nan,
                "solve_seconds"         : np.nan,
                "total_variant_seconds" : np.nan,
                "full_pipeline_seconds" : np.nan,

                "open_facilities_count" : 0 ,
                "open_facilities"       : "",
                "status"                : "PIPELINE_ERROR",
                "error_message"         : error_message   ,
            }
        )

    return rows

In [15]:
def run_single_instance_analysis(
    instance_name      : str,
    *,
    perturbation_seed  : int  ,
    restarts           : int  ,
    max_iter           : int  ,
    factor             : int  ,
    top_fraction       : float,
    details_format     : str  ,
    perturbation_delta : float,
    time_limit_seconds : int | None,
) -> list[dict[str, object]]:
    instance_name = canonical_instance_name(instance_name)
    instance_path = INSTANCES_DIR / instance_name
    instance_id   = Path(instance_name).stem

    metadata        = read_instance_metadata(instance_path)
    best_known_cost = BEST_KNOWN_COSTS.get  (instance_id  )

    try:
        meta_started_at  = perf_counter()
        summary, details = pymedian.solve_pmedian(
            instance_path,
            restarts      = restarts      ,
            max_iter      = max_iter      ,
            factor        = factor        ,
            details_format= details_format,
        )
        metaheuristic_runtime_seconds = perf_counter() - meta_started_at

        structure_started_at = perf_counter()
        insights = extract_structures_and_frequency_candidates(
            summary,
            details,
            top_fraction=top_fraction,
        )
        structure_runtime_seconds = perf_counter() - structure_started_at

        perturbation_started_at = perf_counter()
        original_graph          = read_orlibrary_graph(instance_path)
        perturbed_adjacency, perturbation_stats = perturb_adjacency(
            original_graph["adjacency"],
            delta=perturbation_delta,
            seed =perturbation_seed ,
        )
        perturbed_distances          = all_pairs_shortest_paths(perturbed_adjacency)
        perturbation_runtime_seconds = perf_counter() - perturbation_started_at

        previous_solution_perturbed_objective = compute_solution_cost(
            perturbed_distances,
            list(insights["best_facilities"]),
        )

        common = {
            "instance"        : instance_name    ,
            "instance_id"     : instance_id      ,
            "n"               : int(summary["n"]),
            "p"               : int(summary["p"]),
            "best_known_cost" : best_known_cost if best_known_cost is not None else np.nan,

            "metaheuristic_best_cost"       : insights["best_cost"]        ,
            "metaheuristic_runtime_seconds" : metaheuristic_runtime_seconds,
            "metaheuristic_gap_percent"     : gap_to_reference_percent(insights["best_cost"], best_known_cost),

            "structure_extraction_runtime_seconds" : structure_runtime_seconds   ,
            "perturbation_runtime_seconds"         : perturbation_runtime_seconds,
            "pre_reoptimization_seconds"           : metaheuristic_runtime_seconds + structure_runtime_seconds + perturbation_runtime_seconds,

            "previous_solution_degradation_percent" : gap_to_reference_percent(
                previous_solution_perturbed_objective, insights["best_cost"]
            ),
            "previous_solution_perturbed_objective": (
                previous_solution_perturbed_objective
                if   previous_solution_perturbed_objective is not None
                else np.nan
            ),

            "memory_size"        : insights["memory_size"       ],
            "top_solution_count" : insights["top_solution_count"],
            "top_cost_cutoff"    : insights["top_cost_cutoff"   ],
            "cooccurrence_edges" : insights["cooccurrence_edges"],

            "k_core_max_level"          : insights["k_core_max_level"         ],
            "k_core_candidate_count"    : insights["k_core_candidate_count"   ],
            "k_core_candidate_fraction" : insights["k_core_candidate_fraction"],
            "k_core_best_set_recall"    : insights["k_core_best_set_recall"   ],

            "densest_subgraph_candidate_count"    : insights["densest_subgraph_candidate_count"   ],
            "densest_subgraph_candidate_fraction" : insights["densest_subgraph_candidate_fraction"],
            "densest_subgraph_density"            : insights["densest_subgraph_density"           ],
            "densest_subgraph_best_set_recall"    : insights["densest_subgraph_best_set_recall"   ],

            "nonzero_frequency_count"    : insights["nonzero_frequency_count"   ],
            "nonzero_frequency_fraction" : insights["nonzero_frequency_fraction"],

            "top_frequency_k_core_size_count"           : insights["top_frequency_k_core_size_count"          ],
            "top_frequency_k_core_size_best_set_recall" : insights["top_frequency_k_core_size_best_set_recall"],
            "top_frequency_k_core_k_core_intersection"  : insights["top_frequency_k_core_k_core_intersection" ],

            "top_frequency_densest_size_count"           : insights["top_frequency_densest_size_count"          ],
            "top_frequency_densest_size_best_set_recall" : insights["top_frequency_densest_size_best_set_recall"],
            "top_frequency_densest_densest_intersection" : insights["top_frequency_densest_densest_intersection"],

            "perturbation_delta" : perturbation_delta,
            "perturbation_seed"  : perturbation_seed ,
            **perturbation_stats,
        }

        rows = []

        for spec in VARIANT_SPECS:
            allowed_facilities = None
            if spec["allowed_key"] is not None:
                allowed_facilities = sorted(insights[spec["allowed_key"]])

            row = solve_reoptimization_variant(
                variant            = spec["variant"      ],
                variant_order      = spec["variant_order"],
                distances          = perturbed_distances  ,
                p                  = int(summary["p"]) ,
                time_limit_seconds = time_limit_seconds,
                allowed_facilities = allowed_facilities,
            )

            row.update(common)

            row["improvement_over_previous_percent"] = improvement_percent(
                previous_solution_perturbed_objective,
                row["objective_value"] if pd.notna(row["objective_value"]) else None,
            )

            row["full_pipeline_seconds"] = common["pre_reoptimization_seconds"] + (
                row["total_variant_seconds"]
                if   pd.notna(row["total_variant_seconds"])
                else 0.0
            )

            rows.append(row)

        return rows
    except Exception as exc:
        return build_pipeline_error_rows(
            instance_name  ,
            instance_id    ,
            metadata       ,
            best_known_cost,
            f"{type(exc).__name__}: {exc}",
        )

In [16]:
def run_benchmark(
    instance_names: list[str],
    *,
    restarts           : int  ,
    max_iter           : int  ,
    factor             : int  ,
    top_fraction       : float,
    details_format     : str  ,
    global_seed        : int  ,
    perturbation_delta : float,
    time_limit_seconds : int | None,
) -> pd.DataFrame:
    if not instance_names:
        raise ValueError("The benchmark requires at least one instance.")

    rows = []

    for offset, instance_name in enumerate(
        tqdm(
            instance_names,
            total        =len(instance_names)                 ,
            desc         ="Frequency candidate reoptimization",
            dynamic_ncols=True                                ,
        )
    ):
        rows.extend(
            run_single_instance_analysis(
                instance_name,
                perturbation_seed = global_seed + offset,
                restarts          = restarts          ,
                max_iter          = max_iter          ,
                factor            = factor            ,
                top_fraction      = top_fraction      ,
                details_format    = details_format    ,
                perturbation_delta= perturbation_delta,
                time_limit_seconds= time_limit_seconds,
            )
        )

    results_df = pd.DataFrame(rows)

    results_df["instance_order"] = results_df["instance"].map(
        lambda value: instance_sort_key(value)[0]
    )

    return (
        results_df.sort_values(["instance_order", "instance", "variant_order"])
                  .drop       (columns="instance_order")
                  .reset_index(drop   =True            )
    )

### RESULT TABLES

In [17]:
def add_unrestricted_reference(results_df: pd.DataFrame) -> pd.DataFrame:
    baseline_df = (
        results_df.loc[
            results_df["variant"] == "baseline",
            [
                "instance"       ,
                "status"         ,
                "objective_value",
            ],
        ]
        .rename(
            columns={
                "status"          : "baseline_status"         ,
                "objective_value" : "baseline_objective_value",
            }
        )
    )

    merged = results_df.merge(baseline_df, on="instance", how="left")

    merged["gap_to_unrestricted_percent"] = np.where(
        merged  ["objective_value"         ].notna()
        & merged["baseline_objective_value"].notna(),
        100.0
        * (merged["objective_value"] - merged["baseline_objective_value"])
         / merged["baseline_objective_value"],
        np.nan,
    )

    merged["baseline_is_optimal"] = merged["baseline_status"].eq("OPTIMAL")

    return merged

In [18]:
def build_instance_results_table(results_df: pd.DataFrame) -> pd.DataFrame:
    instance_columns = [
        "instance"       ,
        "instance_id"    ,
        "n"              ,
        "p"              ,
        "best_known_cost",
        "metaheuristic_best_cost"  ,
        "metaheuristic_gap_percent",
        "previous_solution_perturbed_objective"     ,
        "previous_solution_degradation_percent"     ,
        "memory_size"                               ,
        "top_solution_count"                        ,
        "k_core_candidate_count"                    ,
        "densest_subgraph_candidate_count"          ,
        "nonzero_frequency_count"                   ,
        "top_frequency_k_core_size_count"           ,
        "top_frequency_densest_size_count"          ,
        "top_frequency_k_core_k_core_intersection"  ,
        "top_frequency_densest_densest_intersection",
        "pre_reoptimization_seconds"                ,
    ]

    metric_columns = [
        "status"                           ,
        "candidate_facility_count"         ,
        "objective_value"                  ,
        "improvement_over_previous_percent",
        "gap_to_unrestricted_percent"      ,
        "total_variant_seconds"            ,
        "open_facilities"                  ,
    ]

    base_df = results_df[instance_columns].drop_duplicates("instance")

    wide_df         = results_df.pivot(index="instance", columns="variant", values=metric_columns)
    wide_df.columns = [
        f"{variant}__{metric}"
        for metric, variant in wide_df.columns
    ]
    wide_df = wide_df.reset_index()

    return base_df.merge(wide_df, on="instance", how="left")


def build_variant_aggregate(results_df: pd.DataFrame) -> pd.DataFrame:
    feasible_statuses = {"OPTIMAL", "FEASIBLE"}

    return (
        results_df
        .groupby(["variant_order", "variant"], dropna=False)
        .agg(
            instances                         =("instance", "nunique"),
            solved_instances                  =("objective_value", lambda s: int( s.notna()               .sum())),
            optimal_instances                 =("status"         , lambda s: int((s == "OPTIMAL")         .sum())),
            feasible_instances                =("status"         , lambda s: int(s.isin(feasible_statuses).sum())),

            mean_candidate_facility_count     =("candidate_facility_count"         , "mean"),
            mean_candidate_facility_fraction  =("candidate_facility_fraction"      , "mean"),
            mean_objective_value              =("objective_value"                  , "mean"),
            mean_improvement_over_previous_pct=("improvement_over_previous_percent", "mean"),
            mean_gap_to_unrestricted_pct      =("gap_to_unrestricted_percent"      , "mean"),
            mean_total_variant_seconds        =("total_variant_seconds"            , "mean"),
            mean_full_pipeline_seconds        =("full_pipeline_seconds"            , "mean"),
        )
        .reset_index()
        .sort_values("variant_order")
        .reset_index(drop=True      )
    )

### APPLYING

In [19]:
raw_results_df = run_benchmark(
    INSTANCE_NAMES,
    restarts           = RESTARTS          ,
    max_iter           = MAX_ITER          ,
    factor             = FACTOR            ,
    top_fraction       = TOP_FRACTION      ,
    details_format     = DETAILS_FORMAT    ,
    global_seed        = GLOBAL_SEED       ,
    perturbation_delta = PERTURBATION_DELTA,
    time_limit_seconds = TIME_LIMIT_SECONDS,
)

Frequency candidate reoptimization:   0%|          | 0/20 [00:00<?, ?it/s]

In [20]:
results_df           = add_unrestricted_reference  (raw_results_df)
instance_results_df  = build_instance_results_table(results_df)
aggregate_results_df = build_variant_aggregate     (results_df)

failures_df         = results_df[results_df["error_message"].notna()].copy()

print(f"Raw result rows : {len(results_df )}")
print(f"Failures        : {len(failures_df)}")
print(f"Instances       : {results_df['instance'].nunique()}")
print(f"Variants        : {results_df['variant' ].nunique()}")

Raw result rows : 120
Failures        : 0
Instances       : 20
Variants        : 6


Table for instance:

In [21]:
display(instance_results_df[[
    "instance_id"              ,
    "n"                        ,
    "p"                        ,
    "metaheuristic_gap_percent",

    "k_core_candidate_count"          ,
    "densest_subgraph_candidate_count",
    "nonzero_frequency_count"         ,

    "baseline__gap_to_unrestricted_percent"                  ,
    "densest_subgraph__gap_to_unrestricted_percent"          ,
    "highest_k_core__gap_to_unrestricted_percent"            ,
    "nonzero_frequency__gap_to_unrestricted_percent"         ,
    "top_frequency_densest_size__gap_to_unrestricted_percent",
    "top_frequency_k_core_size__gap_to_unrestricted_percent" ,

    "baseline__total_variant_seconds"                  ,
    "densest_subgraph__total_variant_seconds"          ,
    "highest_k_core__total_variant_seconds"            ,
    "nonzero_frequency__total_variant_seconds"         ,
    "top_frequency_densest_size__total_variant_seconds",
    "top_frequency_k_core_size__total_variant_seconds" ,
]].round(4))

,instance_id,n,p,metaheuristic_gap_percent,k_core_candidate_count,densest_subgraph_candidate_count,nonzero_frequency_count,baseline__gap_to_unrestricted_percent,densest_subgraph__gap_to_unrestricted_percent,highest_k_core__gap_to_unrestricted_percent,nonzero_frequency__gap_to_unrestricted_percent,top_frequency_densest_size__gap_to_unrestricted_percent,top_frequency_k_core_size__gap_to_unrestricted_percent,baseline__total_variant_seconds,densest_subgraph__total_variant_seconds,highest_k_core__total_variant_seconds,nonzero_frequency__total_variant_seconds,top_frequency_densest_size__total_variant_seconds,top_frequency_k_core_size__total_variant_seconds
0,pmed1,100,5,0.0000,7,5,7,0.0,0.0,0.0,0.0,0.0,0.0,1.475317,0.009942,0.058446,0.056041,0.008054,0.055674
1,pmed2,100,10,0.0000,12,12,12,0.0,0.0,0.0,0.0,0.0,0.0,2.12477,0.103393,0.094969,0.103201,0.084955,0.091402
2,pmed3,100,10,0.0000,13,10,13,0.0,0.0,0.0,0.0,0.0,0.0,1.841825,0.015366,0.093912,0.095397,0.013438,0.09269
3,pmed4,100,20,0.0000,23,22,23,0.0,0.0,0.0,0.0,0.0,0.0,1.093752,0.15884,0.164233,0.171905,0.154556,0.162661
4,pmed5,100,33,0.0000,37,34,38,0.0,0.0,0.0,0.0,0.0,0.0,1.092012,0.257103,0.304781,0.297738,0.249256,0.287579
5,pmed6,200,5,0.0000,8,6,9,0.0,0.0,0.0,0.0,0.0,0.0,117.273423,0.104618,0.196211,0.159702,0.09517,0.137167
6,pmed7,200,10,0.0000,12,12,12,0.0,0.0,0.0,0.0,0.0,0.0,10.91168,0.19222,0.184958,0.204652,0.180334,0.197941
7,pmed8,200,20,0.0000,22,20,23,0.0,0.0,0.0,0.0,0.0,0.0,9.382673,0.046393,0.355353,0.392219,0.048779,0.348307
8,pmed9,200,40,0.0000,48,43,51,0.0,0.0,0.0,0.0,0.0,0.0,9.117503,0.872973,1.023182,1.120007,0.868715,1.031956
9,pmed10,200,67,0.0000,75,67,82,0.0,1.117318,0.0,0.0,1.117318,0.0,9.106703,0.157364,2.01566,2.192762,0.147992,2.026501


Grouping by variant:

In [22]:
display(aggregate_results_df[[
    "variant"          ,
    "instances"        ,
    "solved_instances" ,
    "optimal_instances",

    "mean_candidate_facility_fraction",
    "mean_gap_to_unrestricted_pct"    ,
    "mean_total_variant_seconds"      ,
]].round(4))

,variant,instances,solved_instances,optimal_instances,mean_candidate_facility_fraction,mean_gap_to_unrestricted_pct,mean_total_variant_seconds
0,baseline,20,19,19,1.0000,0.0000,43.3925
1,highest_k_core,20,20,20,0.1637,0.0052,2.3391
2,densest_subgraph,20,20,20,0.1483,0.0696,1.2910
3,nonzero_frequency,20,20,20,0.1751,0.0030,2.9634
4,top_frequency_k_core_size,20,20,20,0.1637,0.0063,2.3309
5,top_frequency_densest_size,20,20,20,0.1483,0.0696,1.2927


In [23]:
if not failures_df.empty:
    display(
        failures_df[
            [
                "instance"     ,
                "variant"      ,
                "status"       ,
                "error_message",
            ]
        ]
    )

### SAVING RESULTS

In [24]:
if SAVE_RESULTS_CSV:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    results_df          .to_csv(RAW_RESULTS_CSV      , index=False)
    instance_results_df .to_csv(SUMMARY_RESULTS_CSV  , index=False)
    aggregate_results_df.to_csv(AGGREGATE_RESULTS_CSV, index=False)

    print(f"Raw results saved to     : {      RAW_RESULTS_CSV}")
    print(f"Instance table saved to  : {  SUMMARY_RESULTS_CSV}")
    print(f"Aggregate table saved to : {AGGREGATE_RESULTS_CSV}")

    if not failures_df.empty:
        failures_df.to_csv(FAILURES_CSV, index=False)

        print(f"Failures saved to : {FAILURES_CSV}")

Raw results saved to     : /home/rei-luisinho/xopt/notebooks/experiments_sbpo/artifacts/reoptimization_frequency_candidates_raw.csv
Instance table saved to  : /home/rei-luisinho/xopt/notebooks/experiments_sbpo/artifacts/reoptimization_frequency_candidates_summary.csv
Aggregate table saved to : /home/rei-luisinho/xopt/notebooks/experiments_sbpo/artifacts/reoptimization_frequency_candidates_aggregate.csv
